# Catalog EDA Template

Use this notebook after a catalog has been fetched and converted into a processed Parquet dataset.

Suggested scope:
- inspect schema
- review missingness
- summarize photometric and redshift fields
- visualize basic distributions

Out of scope for now:
- anomaly detection
- outlier hunting
- model fitting beyond basic descriptive analysis

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from jwst_galaxy_analysis.config import get_paths

sns.set_theme(context='notebook', style='whitegrid')
paths = get_paths()
dataset_path = next(iter(sorted(paths.data_processed.glob('*.parquet'))), None)
dataset_path

In [ ]:
if dataset_path is None:
    raise FileNotFoundError('No processed datasets found. Fetch a catalog first.')

df = pd.read_parquet(dataset_path)
df.head()

In [ ]:
redshift_candidates = [
    column for column in df.columns
    if 'redshift' in column.lower() or column.lower() in {'z', 'z_phot', 'zspec', 'z_spec', 'zbest'}
]

print(f'Row count: {len(df):,}')
print('Available columns:')
for column in df.columns:
    print(f'  - {column}')

if redshift_candidates:
    redshift_column = redshift_candidates[0]
    print(f'\nUsing redshift column: {redshift_column}')
    display(df[redshift_column].describe())
    sns.histplot(df[redshift_column].dropna(), bins=40)
    plt.title(f'Redshift distribution: {redshift_column}')
    plt.xlabel(redshift_column)
    plt.tight_layout()
else:
    print('\nNo redshift-like column found.')

In [ ]:
summary = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'missing_fraction': df.isna().mean(),
    'unique_values': df.nunique(dropna=True),
})
summary.sort_values('missing_fraction', ascending=False).head(20)

In [ ]:
numeric_columns = df.select_dtypes(include='number').columns.tolist()
numeric_columns[:10]

In [ ]:
if numeric_columns:
    ax = df[numeric_columns[:4]].hist(figsize=(10, 8), bins=40)
    plt.tight_layout()
    plt.savefig(paths.figures / f'{dataset_path.stem}_numeric_overview.png', dpi=150)
else:
    print('No numeric columns available for histogram overview.')